# Assignment 2 — Modelling and Evaluation

This notebook develops a baseline and two supervised classification models to identify crowded train observations. The models will be evaluated consistently using the agreed evaluation metrics, with attention to target leakage and the operational consequences of prediction errors.

## Scope and evaluation contract

### Stakeholder decision
The NSW train operations team needs to identify services that may experience passenger crowding so that capacity and timetable reviews can be prioritised.

### Prediction task
The target is `Crowded`, where 1 indicates `FEW_SEATS_AVAILABLE` or `STANDING_ROOM_ONLY`, and 0 indicates `MANY_SEATS_AVAILABLE`. Each row represents an observation of occupancy at a train station stop. The model uses the available service and train characteristics to classify whether that observation is crowded.

### Decision time
The current dataset contains recorded arrival and departure times. These variables are excluded from the predictors because their availability before the operational decision has not been established. Predictions should only be used operationally after confirming that all selected features would be available at the intended decision time.

### Evaluation metrics
Recall for the crowded class is the primary metric because failing to identify a crowded service could prevent the operations team from investigating a genuine capacity issue. Precision and F1-score for the crowded class are secondary metrics. Accuracy is reported for context but is not sufficient on its own because crowded observations are a minority of the dataset.

### Consequences of prediction errors
A false negative occurs when a crowded observation is classified as not crowded. This could cause a genuine crowding issue to be overlooked. A false positive occurs when a non-crowded observation is classified as crowded, potentially directing staff attention or capacity reviews towards a service that does not need them. The preferred balance depends on the operational cost of missed crowding versus unnecessary investigation.

### Evaluation design
A stratified 80/20 train/test split with `random_state=42` is used to preserve the approximate target-class proportions. Preprocessing is fitted on the training data through a pipeline to avoid using test data to learn imputation or encoding settings. The baseline and candidate models will be evaluated on the same test set using consistent metrics.

### Evaluation metric choice. 
Crowded-class recall is the primary evaluation metric because failing to identify a genuinely crowded train service could leave a capacity problem uninvestigated. Crowded-class precision is a secondary metric because false alarms may lead to unnecessary operational investigations. The crowded-class F1-score provides an additional summary of the precision–recall trade-off, while accuracy is reported for completeness but is not used alone to select a model because the target is imbalanced. These priorities should be confirmed with the operations stakeholder before any operational deployment.
Recall is pre-selected as the primary metric for this experiment because missed crowded services may leave genuine capacity issues uninvestigated. Precision, crowded-class F1-score and accuracy are secondary metrics. This choice should be confirmed with the operations stakeholder before operational deployment, particularly after the costs of false negatives and false positives are assessed.

### Baseline and candidate model design

The majority-class baseline uses `DummyClassifier(strategy="most_frequent")`. It predicts the most common class for every observation and provides a transparent reference against which the candidate models can be evaluated.

Two supervised classification models are compared: Logistic Regression and Random Forest. Logistic Regression provides a relatively simple linear classification approach, while Random Forest combines multiple decision trees and can capture more complex relationships between features. These models provide different approaches to the same binary classification task.

Both candidate models use the same preprocessing pipeline. Missing categorical values are imputed using the most frequent category, and categorical features are one-hot encoded with unknown categories ignored. Missing numerical values are imputed using the median, followed by standardisation. Preprocessing is fitted through the training pipeline to avoid learning preprocessing parameters from the test set.

The train/test split is stratified, with 80% of observations used for training and 20% for testing. `random_state=42` supports reproducibility. Both candidate models use `class_weight="balanced"` to account for the low proportion of crowded observations. Logistic Regression uses `max_iter=1000`; Random Forest uses 200 trees and `n_jobs=-1`.

All three approaches are evaluated on the same test set. The comparison includes accuracy, crowded-class precision, crowded-class recall and crowded-class F1-score. The confusion matrices are also examined to distinguish false alarms from missed crowded observations. This is important because the majority-class baseline can achieve high accuracy while failing to detect any crowded observations.

In [15]:

from pathlib import Path
import pandas as pd
import numpy as np

data_path = Path("../../data/processed/train_occupancy_clean_full.csv")
train_df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Shape:", train_df.shape)
display(train_df.head())
print("\nColumns:")
print(train_df.columns.tolist())


Dataset loaded successfully.
Shape: (49927, 16)


,day,Actual.Stop.Station,Actual.Station.Arrv.Time,Actual.Station.Dprt.Time,Segment.Direction,Trip.Name,Service.Line,Orig..Station,Dest..Station,Leading.Set.Type,Node.Seq.Order,Actual.Station.Dprt.Time.Band,Occupancy Status,Occupancy Range,Arrival_Clean,Departure_Clean
0,9,Miranda,2017-01-10 00:18:17,2017-01-10 00:19:04,Down,620S,Illawarra,Central,Cronulla,T,18,00:15-00:29,MANY_SEATS_AVAILABLE,Low: 0-399,2017-01-10 00:18:17,2017-01-10 00:19:04
1,13,Beecroft,2017-01-13 08:05:59,2017-01-13 08:06:56,Down,152C,North via Macquarie Park,Central,Hornsby,A,16,08:00-08:14,MANY_SEATS_AVAILABLE,Low: 0-399,2017-01-13 08:05:59,2017-01-13 08:06:56
2,12,Arncliffe,2017-01-12 07:22:51,2017-01-12 07:23:36,Down,607B,Illawarra,Central,Waterfall,T,6,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399,2017-01-12 07:22:51,2017-01-12 07:23:36
3,13,Dulwich Hill,2017-01-13 07:20:39,2017-01-13 07:21:53,Down,58-F,Bankstown,Central,Birrong,A,7,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399,2017-01-13 07:20:39,2017-01-13 07:21:53
4,11,Belmore,2017-01-11 18:38:28,2017-01-11 18:39:07,Up,33-P,Bankstown,Birrong,Central,K,7,18:30-18:44,MANY_SEATS_AVAILABLE,Low: 0-399,2017-01-11 18:38:28,2017-01-11 18:39:07



Columns:
['day', 'Actual.Stop.Station', 'Actual.Station.Arrv.Time', 'Actual.Station.Dprt.Time', 'Segment.Direction', 'Trip.Name', 'Service.Line', 'Orig..Station', 'Dest..Station', 'Leading.Set.Type', 'Node.Seq.Order', 'Actual.Station.Dprt.Time.Band', 'Occupancy Status', 'Occupancy Range', 'Arrival_Clean', 'Departure_Clean']


In [16]:
train_df["Occupancy Status"].value_counts(dropna=False)

Occupancy Status
MANY_SEATS_AVAILABLE    47933
FEW_SEATS_AVAILABLE      1595
STANDING_ROOM_ONLY        399
Name: count, dtype: int64

In [17]:
# Create a binary target: 1 = crowded, 0 = not crowded
train_df["Crowded"] = (
    train_df["Occupancy Status"].isin(
        ["FEW_SEATS_AVAILABLE", "STANDING_ROOM_ONLY"]
    ).astype(int)
)

# Check the class balance
print(train_df["Crowded"].value_counts())
print(train_df["Crowded"].value_counts(normalize=True).mul(100).round(2))

Crowded
0    47933
1     1994
Name: count, dtype: int64
Crowded
0    96.01
1     3.99
Name: proportion, dtype: float64


In [18]:
drop_cols = [
    "Occupancy Status",
    "Occupancy Range",
    "Actual.Station.Arrv.Time",
    "Actual.Station.Dprt.Time",
    "Actual.Station.Dprt.Time.Band",
    "Crowded"
]

X = train_df.drop(columns=drop_cols)
y = train_df["Crowded"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Missing values in features:", X.isna().sum().sum())

Feature shape: (49927, 11)
Target shape: (49927,)
Missing values in features: 474


In [19]:
# Check which features have missing values
missing = X.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

Orig..Station          79
Segment.Direction      78
Leading.Set.Type       76
Actual.Stop.Station    72
Dest..Station          63
Service.Line           59
Departure_Clean        28
Arrival_Clean          19
dtype: int64


In [20]:
# Remove timestamp columns from the predictors
X = X.drop(columns=["Arrival_Clean", "Departure_Clean"], errors="ignore")

# Identify categorical and numerical features
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_cols)
print("Numerical features:", numeric_cols)
print("Rows:", len(X))

Categorical features: ['Actual.Stop.Station', 'Segment.Direction', 'Trip.Name', 'Service.Line', 'Orig..Station', 'Dest..Station', 'Leading.Set.Type']
Numerical features: ['day', 'Node.Seq.Order']
Rows: 49927


/var/folders/tr/lmhnglwx56l3q9928kbgw4480000gn/T/ipykernel_35142/2215012923.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True).round(4))

Training rows: 39941
Testing rows: 9986

Training target proportions:
Crowded
0    0.9601
1    0.0399
Name: proportion, dtype: float64

Testing target proportions:
Crowded
0    0.96
1    0.04
Name: proportion, dtype: float64


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_cols = X_train.select_dtypes(include=["object", "str"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_cols),
    ("numeric", numeric_pipeline, numeric_cols)
])

print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numeric_cols))
print("Preprocessing setup complete.")

Categorical features: 7
Numerical features: 2
Preprocessing setup complete.


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9368

Confusion matrix:
[[9000  587]
 [  44  355]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      9587
           1       0.38      0.89      0.53       399

    accuracy                           0.94      9986
   macro avg       0.69      0.91      0.75      9986
weighted avg       0.97      0.94      0.95      9986



In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, rf_pred))
print("\nClassification report:")
print(classification_report(y_test, rf_pred, zero_division=0))

Accuracy: 0.9716

Confusion matrix:
[[9520   67]
 [ 217  182]]

Classification report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      9587
           1       0.73      0.46      0.56       399

    accuracy                           0.97      9986
   macro avg       0.85      0.72      0.77      9986
weighted avg       0.97      0.97      0.97      9986



In [25]:

from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, baseline_pred))
print("\nClassification report:")
print(classification_report(y_test, baseline_pred, zero_division=0))


Baseline accuracy: 0.96

Confusion matrix:
[[9587    0]
 [ 399    0]]

Classification report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      9587
           1       0.00      0.00      0.00       399

    accuracy                           0.96      9986
   macro avg       0.48      0.50      0.49      9986
weighted avg       0.92      0.96      0.94      9986



In [26]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for name, predictions in [
    ("Baseline", baseline_pred),
    ("Logistic Regression", y_pred),
    ("Random Forest", rf_pred)
]:
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Crowded Precision": precision_score(y_test, predictions, zero_division=0),
        "Crowded Recall": recall_score(y_test, predictions, zero_division=0),
        "Crowded F1": f1_score(y_test, predictions, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df.round(4)


,Model,Accuracy,Crowded Precision,Crowded Recall,Crowded F1
0,Baseline,0.9600,0.0000,0.0000,0.0000
1,Logistic Regression,0.9368,0.3769,0.8897,0.5295
2,Random Forest,0.9716,0.7309,0.4561,0.5617


In [27]:

from sklearn.metrics import confusion_matrix

for name, predictions in [
    ("Logistic Regression", y_pred),
    ("Random Forest", rf_pred)
]:
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    print(name)
    print("False alarms:", fp)
    print("Missed crowded observations:", fn)
    print("Correctly identified crowded observations:", tp)
    print()


Logistic Regression
False alarms: 587
Missed crowded observations: 44
Correctly identified crowded observations: 355

Random Forest
False alarms: 67
Missed crowded observations: 217
Correctly identified crowded observations: 182



## Results and stakeholder interpretation

The majority-class baseline achieved 96.00% accuracy but failed to identify any crowded observations. This demonstrates why accuracy alone is insufficient for this imbalanced classification task.

Logistic Regression achieved 93.68% accuracy, 88.97% recall for crowded observations and a crowded-class F1-score of 0.5295. It correctly identified 355 crowded test observations and missed 44, but generated 587 false alarms. This model is more effective at detecting crowded services, although its predictions would require more follow-up investigation.

Random Forest achieved 97.16% accuracy and a crowded-class F1-score of 0.5617, with 73.09% precision and 45.61% recall. It correctly identified 182 crowded observations, missed 217 and generated 67 false alarms. It is more selective when flagging crowding, but its lower recall means that many crowded observations would go undetected.

The models therefore involve a practical trade-off. If the stakeholder prioritises identifying as many crowded services as possible, Logistic Regression is the stronger candidate based on recall. If the stakeholder prioritises reducing false alarms and unnecessary operational investigations, Random Forest may be preferable. The final choice should reflect the relative costs of missed crowding and unnecessary intervention, rather than overall accuracy alone.

These results are preliminary test-set evidence, not proof that either model will generalise to future services. Further validation is needed before operational use.

## Stakeholder recommendation

Based on the current test-set results, Logistic Regression is the stronger candidate when the priority is to identify as many crowded observations as possible. It achieved 88.97% recall for the crowded class, compared with 45.61% for Random Forest. However, Logistic Regression generated 587 false alarms, compared with 67 for Random Forest.

For an initial operational investigation, Logistic Regression could be used to flag services for further review when missed crowding is considered particularly costly. These flags should be treated as prompts for investigation rather than automatic instructions to change train capacity or timetables. Random Forest may be preferable when limiting unnecessary investigations is more important.

Before selecting a model for operational use, the operations stakeholder should confirm the relative costs of missed crowding and false alarms. The current results come from a held-out test set and do not establish performance on future services. Further validation using later time periods and confirmation that all predictors are available at the intended decision time are required.

## Model card: Train crowding classification

**Intended use:** Support preliminary investigation of train services that may experience crowding. Predictions are decision-support signals, not automatic instructions to change capacity or timetables.

**Target and unit of prediction:** The binary `Crowded` outcome for an individual train observation, derived from the recorded occupancy status.

**Data:** The cleaned NSW Train Occupancy dataset. The current experiment uses a stratified 80/20 train/test split with `random_state=42`.

**Models evaluated:** Logistic Regression and Random Forest, compared against a majority-class baseline. Preprocessing is fitted within a pipeline using training data.

**Test-set performance:** Logistic Regression achieved 93.68% accuracy, 88.97% crowded-class recall, 37.69% crowded-class precision and 0.5295 crowded-class F1. Random Forest achieved 97.16% accuracy, 45.61% recall, 73.09% precision and 0.5617 F1. The baseline achieved 96.00% accuracy but detected no crowded observations.

**Recommended interpretation:** Logistic Regression is the stronger candidate when missed crowding is particularly costly. Random Forest is more selective and may be preferable when false alarms are more costly. The final choice depends on stakeholder priorities.

**Important failure modes:** False alarms may trigger unnecessary investigations; missed crowded observations may leave genuine crowding undetected. Performance on the held-out test set does not guarantee performance on future services.

**Limitations and inappropriate uses:** The model should not be treated as proof of future crowding or used to make automatic operational changes. Further validation across later time periods is required, and predictor availability at the intended decision time must be confirmed.

**Monitoring needs:** Before any operational use, assess performance on later observations, monitor crowded-class precision and recall, review false alarms and missed crowding, and check for changes in data quality or service patterns.